# 04 - Evaluation and error analysis

The test set is used here for the first and only time.

## Load the trained model and rebuild the same test set

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report

from neurovision.data import CLASSES
from neurovision.preprocessing import prepare_data, make_dataset, encoder

model = tf.keras.models.load_model("../models/best_model.keras")
height, width = model.input_shape[1:3]

data = prepare_data("../data/raw", size=(width, height))
X_test, y_test, p_test = data["test"]
test_ds = make_dataset(X_test, y_test)
print(X_test.shape)

## Evaluate on the test set (used once)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

## Predictions and classification report (precision, recall, F1-score)

In [ ]:
probs = model.predict(test_ds)
y_pred = probs.argmax(axis=1)
y_true = encoder.transform(y_test)

print(classification_report(y_true, y_pred, target_names=encoder.classes_, digits=3))

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix (test set)")
plt.savefig("../reports/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Correct and incorrect predictions

In [ ]:
def show_examples(indices, title, filename):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), indices[:8]):
        ax.imshow(X_test[i])
        ax.set_title(
            f"True: {encoder.classes_[y_true[i]]}\nPred: {encoder.classes_[y_pred[i]]} ({probs[i].max():.2f})",
            fontsize=9,
        )
    fig.suptitle(title)
    plt.savefig(f"../reports/{filename}", dpi=150, bbox_inches="tight")
    plt.show()


correct = np.where(y_pred == y_true)[0]
wrong = np.where(y_pred != y_true)[0]
print(f"Correct: {len(correct)} | Incorrect: {len(wrong)}")

show_examples(correct, "Correct predictions", "correct_predictions.png")
show_examples(wrong, "Incorrect predictions", "incorrect_predictions.png")

## Which classes get confused

In [ ]:
pairs = {}
for true_idx, pred_idx in zip(y_true[wrong], y_pred[wrong]):
    key = f"{encoder.classes_[true_idx]} -> {encoder.classes_[pred_idx]}"
    pairs[key] = pairs.get(key, 0) + 1

for key, count in sorted(pairs.items(), key=lambda item: -item[1]):
    print(key, count)

## Explainability with Grad-CAM (one test image per class)

In [ ]:
from neurovision.gradcam import make_gradcam_heatmap, overlay_heatmap

fig, axes = plt.subplots(len(CLASSES), 3, figsize=(9, 12))
for row, name in enumerate(encoder.classes_):
    idx = np.where(y_true == row)[0][0]
    img = X_test[idx]
    heatmap, pred_idx = make_gradcam_heatmap(model, img)
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"True: {name}")
    axes[row, 1].imshow(heatmap, cmap="jet")
    axes[row, 1].set_title(f"Heatmap (pred: {encoder.classes_[pred_idx]})")
    axes[row, 2].imshow(overlay_heatmap(img, heatmap))
    axes[row, 2].set_title("Overlay")
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.savefig("../reports/gradcam.png", dpi=150, bbox_inches="tight")
plt.show()

## predict_image: valid and invalid inputs

In [ ]:
from neurovision.predict import predict_image

print(predict_image(p_test[0], model))

for bad_input in ["missing.jpg", "../README.md"]:
    try:
        predict_image(bad_input, model)
    except (FileNotFoundError, ValueError) as error:
        print(type(error).__name__, "->", error)